# Notebook 5: Data Merging and Preprocessing

## Purpose
Merge original 1,043 expert-annotated comments with 7,000 model-annotated comments, link to articles, and create preprocessed versions for topic modeling.

## Overview
- **Input**: validation_1043_comments.csv, annotated_7000_comments.csv, gnm_articles.csv
- **Output**: merged_8043_comments.csv, cleaned_for_LDA.csv, cleaned_for_NMF.csv, raw_for_BERTopic.csv


In [65]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import warnings
warnings.filterwarnings('ignore')

try:
    import spacy
    SPACY_AVAILABLE = True
    try:
        nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])
    except OSError:
        SPACY_AVAILABLE = False
except ImportError:
    SPACY_AVAILABLE = False

try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer
    try:
        nltk.data.find('tokenizers/punkt')
    except LookupError:
        nltk.download('punkt', quiet=True)
    try:
        nltk.data.find('tokenizers/punkt_tab')
    except LookupError:
        try:
            nltk.download('punkt_tab', quiet=True)
        except Exception:
            pass
    try:
        nltk.data.find('corpora/wordnet')
    except LookupError:
        nltk.download('wordnet', quiet=True)
    try:
        nltk.data.find('corpora/stopwords')
    except LookupError:
        nltk.download('stopwords', quiet=True)
    
    NLTK_AVAILABLE = True
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))
except ImportError:
    NLTK_AVAILABLE = False
except Exception as e:
    NLTK_AVAILABLE = False

base_path = Path('../archive/SOCC')
original_annotated_path = Path('../data/processed/validation_1043_comments.csv')
new_annotated_path = Path('../data/processed/annotated_7000_comments.csv')
articles_path = base_path / 'raw/gnm_articles.csv'
output_dir = Path('../data/processed')
preprocessed_dir = Path('../data/preprocessed')
preprocessed_dir.mkdir(parents=True, exist_ok=True)


## Section 1: Load and Merge Datasets


In [66]:
df_original = pd.read_csv(original_annotated_path)
df_new = pd.read_csv(new_annotated_path)

original_cols = set(df_original.columns)
new_cols = set(df_new.columns)
common_cols = original_cols & new_cols

print(f"Loaded {len(df_original):,} original comments, {len(df_new):,} new comments")


Loaded 1,043 original comments, 7,000 new comments


In [67]:
df_original.columns

Index(['article_id', 'comment_counter', 'title', 'globe_url', 'url',
       'comment_text', 'is_constructive', 'is_constructive:confidence',
       'toxicity_level', 'toxicity_level:confidence',
       'did_you_read_the_article', 'did_you_read_the_article:confidence',
       'annotator_comments', 'expert_is_constructive', 'expert_toxicity_level',
       'expert_comments', 'Labeled'],
      dtype='object')

In [68]:
# Analyze toxicity_level distribution in validation data
if 'toxicity_level' in df_original.columns:
    # Ensure toxicity_level is numeric
    df_original['toxicity_level'] = pd.to_numeric(df_original['toxicity_level'], errors='coerce')
    
    # Get value counts
    toxicity_counts = df_original['toxicity_level'].value_counts().sort_index()
    
    # Calculate percentages
    total_valid = df_original['toxicity_level'].notna().sum()
    missing_count = df_original['toxicity_level'].isna().sum()
    
    print("Toxicity Level Distribution (Validation Data):")
    print("=" * 50)
    print(f"\nCounts:")
    print(toxicity_counts)
    
    print(f"\nPercentages (of valid labels):")
    for level in [1, 2, 3, 4]:
        count = toxicity_counts.get(float(level), toxicity_counts.get(level, 0))
        if total_valid > 0:
            pct = (count / total_valid) * 100
            print(f"Level {level}: {count:,} ({pct:.2f}%)")
    
    print(f"\nMissing values: {missing_count:,}")
    print(f"Total comments: {len(df_original):,}")
    print(f"Valid labels: {total_valid:,}")
else:
    print("'toxicity_level' column not found in validation data")

Toxicity Level Distribution (Validation Data):

Counts:
toxicity_level
1    752
2    217
3     58
4     16
Name: count, dtype: int64

Percentages (of valid labels):
Level 1: 752 (72.10%)
Level 2: 217 (20.81%)
Level 3: 58 (5.56%)
Level 4: 16 (1.53%)

Missing values: 0
Total comments: 1,043
Valid labels: 1,043


In [69]:
df_new.columns

Index(['article_id', 'comment_counter', 'comment_id', 'comment_text',
       'comment_author', 'timestamp', 'datetime', 'year', 'parentID',
       'is_top_level', 'TotalVotes', 'posVotes', 'negVotes', 'vote',
       'engagement_bin', 'threadID', 'replies', 'descendantsCount',
       'sender_isSelf', 'predicted_toxicity_score', 'predicted_toxicity_level',
       'prediction_confidence'],
      dtype='object')

In [70]:
df_original.columns

Index(['article_id', 'comment_counter', 'title', 'globe_url', 'url',
       'comment_text', 'is_constructive', 'is_constructive:confidence',
       'toxicity_level', 'toxicity_level:confidence',
       'did_you_read_the_article', 'did_you_read_the_article:confidence',
       'annotator_comments', 'expert_is_constructive', 'expert_toxicity_level',
       'expert_comments', 'Labeled'],
      dtype='object')

In [71]:
if 'toxicity_level' in df_original.columns:
    df_original = df_original.rename(columns={
        'toxicity_level': 'toxicity_level',
        'is_constructive': 'is_constructive_expert'
    })

if 'predicted_toxicity_level' in df_new.columns:
    df_new = df_new.rename(columns={
        'predicted_toxicity_level': 'toxicity_level',
        'predicted_toxicity_score': 'toxicity_score_model',
        'prediction_confidence': 'toxicity_confidence_model'
    })

df_original['annotation_source'] = 'expert'
df_new['annotation_source'] = 'model'


In [72]:
if 'comment_counter' in common_cols:
    merge_on = 'comment_counter'
elif 'comment_id' in common_cols:
    merge_on = 'comment_id'
else:
    merge_on = None

if merge_on:
    original_duplicates = df_original[merge_on].duplicated().sum()
    new_duplicates = df_new[merge_on].duplicated().sum()
    
    if original_duplicates == 0 and new_duplicates == 0:
        df_merged = pd.merge(
            df_original, 
            df_new, 
            on=merge_on, 
            how='outer',
            suffixes=('_original', '_new')
        )
    else:
        df_merged = pd.concat([df_original, df_new], ignore_index=True, sort=False)
else:
    df_merged = pd.concat([df_original, df_new], ignore_index=True, sort=False)

print(f"Merged dataset: {len(df_merged):,} comments")


Merged dataset: 8,037 comments


In [73]:
df_merged.columns


Index(['article_id_original', 'comment_counter', 'title', 'globe_url', 'url',
       'comment_text_original', 'is_constructive_expert',
       'is_constructive:confidence', 'toxicity_level_original',
       'toxicity_level:confidence', 'did_you_read_the_article',
       'did_you_read_the_article:confidence', 'annotator_comments',
       'expert_is_constructive', 'expert_toxicity_level', 'expert_comments',
       'Labeled', 'annotation_source_original', 'article_id_new', 'comment_id',
       'comment_text_new', 'comment_author', 'timestamp', 'datetime', 'year',
       'parentID', 'is_top_level', 'TotalVotes', 'posVotes', 'negVotes',
       'vote', 'engagement_bin', 'threadID', 'replies', 'descendantsCount',
       'sender_isSelf', 'toxicity_score_model', 'toxicity_level_new',
       'toxicity_confidence_model', 'annotation_source_new'],
      dtype='object')

In [74]:
print(df_new.head())

   article_id         comment_counter                        comment_id  \
0     9317467    source1_9317467_13_1  c9291b0961d644b9abb9a951fb80c53f   
1    13581949   source1_13581949_72_2  e8c42cc8475b4613a4b7408008384ce9   
2     8114824    source1_8114824_38_3  91857efaa9ed42aeabbb1480a6391e37   
3    13216412   source1_13216412_17_2  7c87230963cd4feeb5a2a4f6cff392b9   
4    12292515  source1_12292515_154_6  5157ff4ced8e4c9f968cc32fd66cd00b   

                                        comment_text comment_author  \
0  but would Canada want you? Last week you were ...   southcoaster   
1  That's a bit of a tall order David, Ted's stat...        Matt99f   
2  One of GlynnMhor's irritating little tics is h...     Mark Shore   
3  Who says Hispanics are not racist? It is racis...   Rifleman1010   
4  @newspapertaxi - 'The unmentionable race/class...     freespeech   

      timestamp                 datetime  year  \
0  1.362620e+12  2013-03-07 01:40:08.000  2013   
1  1.375556e+12  2013-

In [75]:
columns_to_drop = []
if 'article_id_original' in df_merged.columns and 'article_id_new' in df_merged.columns:
    df_merged['article_id'] = df_merged['article_id_original'].fillna(df_merged['article_id_new'])
    columns_to_drop.extend(['article_id_original', 'article_id_new'])
elif 'article_id_original' in df_merged.columns:
    df_merged['article_id'] = df_merged['article_id_original']
    columns_to_drop.append('article_id_original')
elif 'article_id_new' in df_merged.columns:
    df_merged['article_id'] = df_merged['article_id_new']
    columns_to_drop.append('article_id_new')

if 'comment_text_original' in df_merged.columns and 'comment_text_new' in df_merged.columns:
    df_merged['comment_text'] = df_merged['comment_text_original'].fillna(df_merged['comment_text_new'])
    columns_to_drop.extend(['comment_text_original', 'comment_text_new'])
elif 'comment_text_original' in df_merged.columns:
    df_merged['comment_text'] = df_merged['comment_text_original']
    columns_to_drop.append('comment_text_original')
elif 'comment_text_new' in df_merged.columns:
    df_merged['comment_text'] = df_merged['comment_text_new']
    columns_to_drop.append('comment_text_new')

if columns_to_drop:
    df_merged = df_merged.drop(columns=columns_to_drop)



## Section 2: Link to Articles


In [76]:
df_articles = pd.read_csv(articles_path)
print(f"Loaded {len(df_articles):,} articles")


Loaded 10,339 articles


In [77]:
df_merged.head()

,comment_counter,title,globe_url,url,is_constructive_expert,is_constructive:confidence,toxicity_level_original,toxicity_level:confidence,did_you_read_the_article,did_you_read_the_article:confidence,...,threadID,replies,descendantsCount,sender_isSelf,toxicity_score_model,toxicity_level_new,toxicity_confidence_model,annotation_source_new,article_id,comment_text
0,UNKNOWN,President Trump will make chaos the new normal,http://www.theglobeandmail.com/opinion/preside...,http://www.sfu.ca/content/dam/sfu/discourse-la...,no,0.9258,1.0,0.4964\n0.3586,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33010454.0,".More petulant commentary from yet another, um..."
1,source1_10012655_7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,2.0,False,0.062236,1.0,1.0,model,10012655.0,TFWs work against the tenets of the free marke...
2,source1_10019010_17_2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5ded3af269a04b4880595ab426d4adf1,NaN,NaN,False,0.080042,1.0,1.0,model,10019010.0,The provinces waste money. They take the money...
3,source1_10030963_101_4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,78348e56c0984d65a78d0d194742b30a,NaN,NaN,False,0.171118,1.0,1.0,model,10030963.0,banking sick days is intelectually dishonest.....
4,source1_10030963_104_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,211ad53b9eba4aee836d9cfd3a882b72,NaN,NaN,False,0.036163,1.0,1.0,model,10030963.0,"Sburg: please explain how this dosn't exist, o..."


In [78]:
if 'article_id' in df_merged.columns and 'article_id' in df_articles.columns:
    article_cols = ['article_id', 'title', 'article_text', 'author', 'published_date']
    article_cols = [col for col in article_cols if col in df_articles.columns]
    
    df_articles_subset = df_articles[article_cols].copy()
    
    df_merged = pd.merge(
        df_merged,
        df_articles_subset,
        on='article_id',
        how='left'
    )
    
    if 'title_x' in df_merged.columns and 'title_y' in df_merged.columns:
        df_merged['title'] = df_merged['title_y'].fillna(df_merged['title_x'])
        df_merged = df_merged.drop(columns=['title_x', 'title_y'])
    elif 'title_x' in df_merged.columns:
        df_merged['title'] = df_merged['title_x']
        df_merged = df_merged.drop(columns=['title_x'])
    elif 'title_y' in df_merged.columns:
        df_merged['title'] = df_merged['title_y']
        df_merged = df_merged.drop(columns=['title_y'])


In [79]:
# Create unified toxicity_level column from merged columns
columns_to_drop_toxicity = []

if 'toxicity_level_original' in df_merged.columns and 'toxicity_level_new' in df_merged.columns:
    # Combine both columns (fillna to use whichever is available)
    df_merged['toxicity_level'] = df_merged['toxicity_level_new'].fillna(df_merged['toxicity_level_original'])
    columns_to_drop_toxicity.extend(['toxicity_level_original', 'toxicity_level_new'])
elif 'toxicity_level_original' in df_merged.columns:
    df_merged['toxicity_level'] = df_merged['toxicity_level_original']
    columns_to_drop_toxicity.append('toxicity_level_original')
elif 'toxicity_level_new' in df_merged.columns:
    df_merged['toxicity_level'] = df_merged['toxicity_level_new']
    columns_to_drop_toxicity.append('toxicity_level_new')

# Drop other toxicity-related columns
for col in ['toxicity_score_model', 'toxicity_confidence_model', 'toxicity_level:confidence']:
    if col in df_merged.columns and col not in columns_to_drop_toxicity:
        columns_to_drop_toxicity.append(col)

if columns_to_drop_toxicity:
    df_merged = df_merged.drop(columns=columns_to_drop_toxicity)

merged_output_path = output_dir / 'merged_8043_comments.csv'
df_merged.to_csv(merged_output_path, index=False)
print(f"Saved merged dataset: {merged_output_path} ({len(df_merged):,} comments)")


Saved merged dataset: ..\data\processed\merged_8043_comments.csv (8,037 comments)


## Section 3: Text Preprocessing


In [80]:
def clean_text_basic(text):
    if pd.isna(text) or text == '':
        return ''
    text = str(text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def lemmatize_text(text, use_spacy=True):
    if pd.isna(text) or text == '':
        return ''
    text = str(text)
    if use_spacy and SPACY_AVAILABLE and nlp is not None:
        doc = nlp(text.lower())
        lemmatized = [token.lemma_ for token in doc if not token.is_punct and not token.is_space]
        return ' '.join(lemmatized)
    elif NLTK_AVAILABLE:
        from nltk.tokenize import word_tokenize
        tokens = word_tokenize(text.lower())
        lemmatized = [lemmatizer.lemmatize(token) for token in tokens if token.isalnum()]
        return ' '.join(lemmatized)
    else:
        return text.lower()

def preprocess_for_lda(text, remove_stopwords=True):
    text = clean_text_basic(text)
    if text == '':
        return ''
    text = lemmatize_text(text)
    if text == '':
        return ''
    if remove_stopwords and NLTK_AVAILABLE:
        tokens = text.split()
        tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
        text = ' '.join(tokens)
    return text


In [81]:
df_merged['comment_text_original'] = df_merged['comment_text'].copy()

df_merged['comment_text'] = df_merged['comment_text'].apply(
    lambda x: preprocess_for_lda(x, remove_stopwords=True)
)

if 'article_text' in df_merged.columns:
    df_merged['article_text'] = df_merged['article_text'].apply(clean_text_basic)


In [82]:
if 'toxicity_level' in df_merged.columns:
    print("Toxicity level distribution:")
    print(df_merged['toxicity_level'].value_counts().sort_index())
else:
    print("'toxicity_level' column not found in df_merged")

Toxicity level distribution:
toxicity_level
1.0    6196
2.0    1494
3.0     274
4.0      61
Name: count, dtype: int64


In [89]:
before_count = len(df_merged)
df_base = df_merged.copy()


if 'comment_id' in df_base.columns:
    df_base = df_base[df_base['comment_id'].notna() & (df_base['comment_id'].astype(str).str.strip() != '')].copy()

base_cols = ['article_id', 'comment_counter', 'comment_id', 'comment_text', 'comment_text_original',
            'toxicity_level', 'annotation_source', 'year', 
            'comment_author', 'timestamp', 'datetime', 'parentID', 
            'is_top_level', 'threadID']

article_metadata_cols = ['title', 'article_text', 'author', 'published_date']
for col in article_metadata_cols:
    if col in df_base.columns and col not in base_cols:
        base_cols.append(col)

base_cols = [col for col in base_cols if col in df_base.columns]
df_base = df_base[base_cols].copy()

df_base = df_base[df_base['comment_text'].notna() & (df_base['comment_text'].str.strip() != '')].copy()

cleaned_cols = [col for col in base_cols if col != 'comment_text_original']
df_cleaned = df_base[cleaned_cols].copy()
cleaned_output_path = preprocessed_dir / 'cleaned_comments_with_articles.csv'
df_cleaned.to_csv(cleaned_output_path, index=False)
print(f"Saved cleaned dataset: {cleaned_output_path} ({len(df_cleaned):,} comments)")



Saved cleaned dataset: ..\data\preprocessed\cleaned_comments_with_articles.csv (6,987 comments)


In [91]:
if 'toxicity_level' in df_cleaned.columns:
    print("Toxicity level distribution:")
    print(df_cleaned['toxicity_level'].value_counts().sort_index())
else:
    print("'toxicity_level' column not found in df_merged")

Toxicity level distribution:
toxicity_level
1.0    5440
2.0    1276
3.0     217
4.0      45
Name: count, dtype: int64


## Section 4: Summary


In [85]:
data_dictionary = {
    'merged_8043_comments.csv': {
        'description': 'Complete merged dataset with all 8,043 comments (1,043 expert + 7,000 model annotated)',
        'location': str(output_dir / 'merged_8043_comments.csv'),
    },
}

import json
dict_output_path = output_dir / 'data_dictionary.json'
with open(dict_output_path, 'w') as f:
    json.dump(data_dictionary, f, indent=2)


In [86]:
if 'toxicity_level' in df_merged.columns:
    print("Toxicity level distribution:")
    print(df_merged['toxicity_level'].value_counts().sort_index())
else:
    print("'toxicity_level' column not found in df_merged")

Toxicity level distribution:
toxicity_level
1.0    6196
2.0    1494
3.0     274
4.0      61
Name: count, dtype: int64


In [87]:
df_merged.columns

Index(['comment_counter', 'globe_url', 'url', 'is_constructive_expert',
       'is_constructive:confidence', 'did_you_read_the_article',
       'did_you_read_the_article:confidence', 'annotator_comments',
       'expert_is_constructive', 'expert_toxicity_level', 'expert_comments',
       'Labeled', 'annotation_source_original', 'comment_id', 'comment_author',
       'timestamp', 'datetime', 'year', 'parentID', 'is_top_level',
       'TotalVotes', 'posVotes', 'negVotes', 'vote', 'engagement_bin',
       'threadID', 'replies', 'descendantsCount', 'sender_isSelf',
       'annotation_source_new', 'article_id', 'comment_text', 'article_text',
       'author', 'published_date', 'title', 'toxicity_level',
       'comment_text_original'],
      dtype='object')

In [88]:

if 'toxicity_level' in df_merged.columns:
    toxicity_summary = df_merged.groupby('toxicity_level').agg({
        'comment_counter': 'count',
    }).rename(columns={'comment_counter': 'count'})
    
    numeric_cols = df_merged.select_dtypes(include=[np.number]).columns.tolist()
    if 'toxicity_level' in numeric_cols:
        numeric_cols.remove('toxicity_level')
    
    if len(numeric_cols) > 0:
        means = df_merged.groupby('toxicity_level')[numeric_cols].mean()
        toxicity_summary = pd.concat([toxicity_summary, means], axis=1)
    
    total_comments = toxicity_summary['count'].sum()
    toxicity_summary['percentage'] = (toxicity_summary['count'] / total_comments * 100).round(2)
    
    cols = ['count', 'percentage'] + [c for c in toxicity_summary.columns if c not in ['count', 'percentage']]
    toxicity_summary = toxicity_summary[cols]
    
    print("Summary by Toxicity Level:")
    print(toxicity_summary.to_string())
    
    overall_mean_toxicity = df_merged['toxicity_level'].mean()
    print(f"\nOverall Mean Toxicity Level: {overall_mean_toxicity:.2f}")
    print(f"Total comments: {total_comments:,}")


Summary by Toxicity Level:
                count  percentage  is_constructive:confidence  did_you_read_the_article  did_you_read_the_article:confidence  expert_toxicity_level     timestamp         year  TotalVotes  posVotes  negVotes  replies  descendantsCount    article_id
toxicity_level                                                                                                                                                                                                                                         
1.0              6196       77.21                    0.877455                       1.0                              0.99955               1.186813  1.409131e+12  2014.193063    3.117453  6.646488  1.860101      NaN          2.762364  2.076179e+07
2.0              1494       18.62                    0.860590                       1.0                              1.00000               2.022727  1.408005e+12  2014.165231    3.960063  8.822560  2.485640      NaN          2.98